In [75]:
import os, subprocess
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

MODEL = os.environ.get("MODEL_ID")

In [76]:
TOOLS = [
    {
        "name": "bash",
        "description": "Run a shell command",
        "input_schema": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string"
                }
            },
            "required": ["command"]
        }
    }
]

In [77]:
query1 = "How many .md files are in the current directory?"

In [78]:
messages = [
    {
        "role": "user",
        "content": query1
    }
]

messages

[{'role': 'user',
  'content': 'How many .md files are in the current directory?'}]

In [79]:
response = client.messages.create(model = MODEL, max_tokens = 1000, tools = TOOLS, messages = messages)

### Tasks

#### 1. Find the tool_use block. Read its id and input["command"]

In [80]:
print(response.content[0].type)
print(response.content[0].id)
print(response.content[0].name)
print(response.content[0].input["command"])
print(response.content)

tool_use
toolu_01Nbj2SRaUkWcFc1NU6F2UCr
bash
ls -1a -- *.md .*.md 2>/dev/null | wc -l; ls -a
[ToolUseBlock(id='toolu_01Nbj2SRaUkWcFc1NU6F2UCr', caller=DirectCaller(type='direct'), input={'command': 'ls -1a -- *.md .*.md 2>/dev/null | wc -l; ls -a'}, name='bash', type='tool_use', toolset_name=None)]


#### 2. Run that command in a terminal and copy the output.

#### 3. Append the model's turn to messages with the assistant role, then append a user message containing one tool_result. It needs the matching tool_use_id and the output as content.

In [81]:
messages.append(
    {
        "role": "assistant",
        "content": response.content
    }
)

messages

[{'role': 'user',
  'content': 'How many .md files are in the current directory?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01Nbj2SRaUkWcFc1NU6F2UCr', caller=DirectCaller(type='direct'), input={'command': 'ls -1a -- *.md .*.md 2>/dev/null | wc -l; ls -a'}, name='bash', type='tool_use', toolset_name=None)]}]

In [82]:
command = response.content[0].input["command"]
command

'ls -1a -- *.md .*.md 2>/dev/null | wc -l; ls -a'

In [83]:
result = subprocess.run(command, shell=True, cwd=os.getcwd(),
                           capture_output=True, text=True, timeout=120).stdout
result

'       1\n\x1b\x1b.\x1b[m\x1b[m\n\x1b\x1b..\x1b[m\x1b[m\ns01_agent_loop.md\ns01_python_REPL.ipynb\n'

In [84]:
messages.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": response.content[0].id,
                "content": result
            }
        ]
    }
)

messages

[{'role': 'user',
  'content': 'How many .md files are in the current directory?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01Nbj2SRaUkWcFc1NU6F2UCr', caller=DirectCaller(type='direct'), input={'command': 'ls -1a -- *.md .*.md 2>/dev/null | wc -l; ls -a'}, name='bash', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01Nbj2SRaUkWcFc1NU6F2UCr',
    'content': '       1\n\x1b\x1b.\x1b[m\x1b[m\n\x1b\x1b..\x1b[m\x1b[m\ns01_agent_loop.md\ns01_python_REPL.ipynb\n'}]}]

#### 4. Call client.messages.create(...) again and look at the new stop_reason.

In [85]:
response = client.messages.create(model = MODEL, max_tokens = 1000, tools = TOOLS, messages = messages)

In [86]:
response

Message(id='msg_011ChQkii3MiXChyTYuem55x', container=None, content=[ThinkingBlock(signature='CAQSxgUKEAgUGAI4AUIIdGhpbmtpbmcSDKW32Ppd2amD8nCn4RoMD/e1NxaSLz1uUWgmIjDdQtLR9yqt4d7JVv/Q1DW0FLMKNC8IR3JyMXjcpOLogKhSJaEPcVeqz6KIH8JS2UIq4wThY5stlwZWAS0hgapwNp+YVuTbWhcnZp0szhdbDp2tRccmyJRvgrsvqkVoIT/o6JhcdAu5BPqpGnrdon0t6Co8W0pkXQ4NkzTZXygl+YyTr9YWPn6qoIztNjNXuXJd6DR+Tm/XnNG9s8C1BJvzyPB6HJmRNvecDl1Op3HVt8oYZi34p2n9607gPJb/bpcYalgOjv4hFqikKSDSLzmQCwao+VrOGSPfHDDcGCmxfFFfmtMDKN4oao+xXfgYGSG6SxiMho3SLbila+HeqMB3sUm25kgbKPEmDKGLRjIxSWVK6MJjyKFLBWtQgp6KDaet1A5oUNSfyLLP0GojeFXAHSddYmAjAvhr/GZ3YERo1JSwRhIUhV3wvCE4ZB8NXaQog8U6p9WR5h4fIfnCznxqenZvlr6E/CLR9wwr+ac7Qj4LAch1fhbXWKVqxUQbdcsEY+Ne7Au845iL7jrZ0UJkRhoyxVFnNUAF+FdnCFx2llSkMXfrPWMifl6ii9DXpUFJ5DO6w73Gzmsa/e1F1IjsArUx/ZaybsdemwOvcGr2c2Cs7IKbQPzuqlnmHMsig4uQoaMLf8FT0bG+NvQ3ZsHbKHm+6axnHWaUolW35eDwyy9yekVrGXNfqm6okWknbZaMPl1GjBIuzTSx8wsmM9yvk5CZj6HL9vGYB8u09icDYVDEkky8k14HU4h6GD9dbpJzwTwsnIRoqyrd75JTmpfqM3j0RtWM6acgIYclneFif+sP7Fd8Qn3NjIHybgOsJgTA5x0

In [87]:
print(response.content[0])
print(response.content[1])

ThinkingBlock(signature='CAQSxgUKEAgUGAI4AUIIdGhpbmtpbmcSDKW32Ppd2amD8nCn4RoMD/e1NxaSLz1uUWgmIjDdQtLR9yqt4d7JVv/Q1DW0FLMKNC8IR3JyMXjcpOLogKhSJaEPcVeqz6KIH8JS2UIq4wThY5stlwZWAS0hgapwNp+YVuTbWhcnZp0szhdbDp2tRccmyJRvgrsvqkVoIT/o6JhcdAu5BPqpGnrdon0t6Co8W0pkXQ4NkzTZXygl+YyTr9YWPn6qoIztNjNXuXJd6DR+Tm/XnNG9s8C1BJvzyPB6HJmRNvecDl1Op3HVt8oYZi34p2n9607gPJb/bpcYalgOjv4hFqikKSDSLzmQCwao+VrOGSPfHDDcGCmxfFFfmtMDKN4oao+xXfgYGSG6SxiMho3SLbila+HeqMB3sUm25kgbKPEmDKGLRjIxSWVK6MJjyKFLBWtQgp6KDaet1A5oUNSfyLLP0GojeFXAHSddYmAjAvhr/GZ3YERo1JSwRhIUhV3wvCE4ZB8NXaQog8U6p9WR5h4fIfnCznxqenZvlr6E/CLR9wwr+ac7Qj4LAch1fhbXWKVqxUQbdcsEY+Ne7Au845iL7jrZ0UJkRhoyxVFnNUAF+FdnCFx2llSkMXfrPWMifl6ii9DXpUFJ5DO6w73Gzmsa/e1F1IjsArUx/ZaybsdemwOvcGr2c2Cs7IKbQPzuqlnmHMsig4uQoaMLf8FT0bG+NvQ3ZsHbKHm+6axnHWaUolW35eDwyy9yekVrGXNfqm6okWknbZaMPl1GjBIuzTSx8wsmM9yvk5CZj6HL9vGYB8u09icDYVDEkky8k14HU4h6GD9dbpJzwTwsnIRoqyrd75JTmpfqM3j0RtWM6acgIYclneFif+sP7Fd8Qn3NjIHybgOsJgTA5x0giGYNo7WGG+qHVBpsD1GefzzRWkcpsV0+Ib+wWOgidfyxYxgB', thinking='', typ

In [88]:
print(response.stop_reason)
print(response.content[1].text)


end_turn
The current directory has **1** `.md` file: `s01_agent_loop.md`. I included hidden files in the check. The only other file there is `s01_python_REPL.ipynb`, which is a notebook, not a `.md` file.


In [89]:
query2 = "Whats the capital of France?"

In [90]:
messages = [
    {
        "role": "user",
        "content": query2
    }
]

messages

[{'role': 'user', 'content': 'Whats the capital of France?'}]

In [91]:
response = client.messages.create(model = MODEL, max_tokens = 1000, tools = TOOLS, messages = messages)

In [92]:
print(response.content[0].type)
print(response.content)

text
[TextBlock(citations=None, text='The capital of France is Paris.', type='text')]


In [93]:
print(response.stop_reason)
print(response.content[0].text)

end_turn
The capital of France is Paris.
